In [1]:
import os
import json
import asyncio

In [2]:
from typing import Annotated, Sequence, TypedDict

## Prepare the vectorstore

In [4]:
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [19]:
from langchain_google_vertexai import VertexAIEmbeddings

embedding = VertexAIEmbeddings(model="gemini-embedding-001")

C:\Users\andrewchan\AppData\Local\Temp\ipykernel_24064\955259049.py:3: DeprecationWarning: Use [`GoogleGenerativeAIEmbeddings`][langchain_google_genai.GoogleGenerativeAIEmbeddings] instead.
  embedding = VertexAIEmbeddings(model="gemini-embedding-001")


In [23]:
UK_DESTINATIONS = [ #A
    "Cornwall",
    "North_Cornwall",
    "South_Cornwall",
    "West_Cornwall",
]

persist_directory = "./chroma_langchain_db"

In [24]:
async def build_vectorstore(destinations: Sequence[str]) -> Chroma:
    """Download Wikipedia pages and create a Chroma vector store."""
    urls =["https://en.wikipedia.org/wiki/{slug}" for slug in destinations]
    loader = AsyncHtmlLoader(urls)
    print("Downloading destination pages ...")
    docs = await loader.aload()

    splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=128)
    chunks = sum([splitter.split_documents([doc]) for doc in docs], [])

    print(f"Embedding {len(chunks)} chunks ...")
    vectordb_client = Chroma.from_documents(chunks, embedding=embedding, persist_directory=persist_directory)
    print("Vector store ready.\n")
    return vectordb_client

In [ ]:
ti_vectorstore_client = await build_vectorstore(UK_DESTINATIONS)
ti_vectorstore_client.persist()
### restore vectordb by
#vectordb = Chroma(
#    persist_directory=persist_directory,
#    embedding_function=embedding
#)
ti_retriever = ti_vectorstore_client.as_retriever()

## Set up tools for LLM

In [28]:
from langchain_core.tools import tool

In [57]:
@tool #A
def search_travel_info(query: str) -> str: #B
    """Search embedded WikiVoyage content for information about destinations in England."""
    docs = ti_retriever.invoke(query) #C
    top = docs[:4] if isinstance(docs, list) else docs #C
    return "\n---\n".join(d.page_content for d in top) #D

In [58]:
tools = [search_travel_info]

In [59]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite", # "gemini-2.5-flash-lite" for cheaper use, "gemini-2.5-flash" for normal use
    vertexai=True
)

llm_with_tools = llm.bind_tools(tools)

## Set up State for agent flow

In [60]:
import json
import operator
from typing import Annotated, Sequence, TypedDict

from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage

In [61]:
### The agent state only contains LLM messages, which are appended to the list of messages
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

## Implment tool execution node
The tool execution logic is implemented in a node that examines the LLM's most recent message, extracts any tool calls requested by the model, invokes the corresponding function with provided arguments

In [49]:
### we implement our own ToolsExecutionNode
### langGraph provides a build-in class `ToolNode` which performs the same function

class ToolsExecutionNode:
    """Execute tools requested by the LLM in the last AIMessage."""

    def __init__(self, tools: Sequence):
        self._tools_by_name = {tool.name:tool for tool in tools}

    def __call__(self, state: dict):
        messages: Sequence[BaseMessage] = state.get("messages", [])

        last_msg = messages[-1]
        tool_messages: list[ToolMessage] = []
        tool_calls = getattr(last_msg, "tool_calls", [])

        for tool_call in tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool = self._tools_by_name[tool_name]
            result = tool.invoke(tool_args)
            tool_messages.append(
                ToolMessage(
                    content=json.dumps(result),
                    name=tool_name,
                    tool_call_id=tool_call["id"],
                )
            )
        return {"message": tool_messages}


tools_execution_node = ToolsExecutionNode(tools)

In [62]:
from langgraph.prebuilt import ToolNode

tools_execution_node = ToolNode(tools)

##  Implement LLM node
Add LLM node which coordinates reasoning and action

In [63]:
def llm_node(state: AgentState):
    """LLM node that decides whether to call the search tool."""
    current_messages = state["messages"]
    response_message = llm_with_tools.invoke(current_messages)
    return {"messages": [response_message]}

## Assemble the agent graph

In [64]:
from langgraph.prebuilt import tools_condition
from langgraph.graph import StateGraph, END

In [65]:
builder = StateGraph(AgentState)

builder.add_node("llm_node", llm_node)
builder.add_node("tools", tools_execution_node)

builder.add_conditional_edges("llm_node", tools_condition)

builder.add_edge("tools", "llm_node")

builder.set_entry_point("llm_node")

travel_info_agent = builder.compile()

## Chat Loop

In [69]:
print("UK Travel Assistant (type 'exit' to quit)")
while True:
    user_input = input("You: ").strip()
    if user_input.lower() in {"exit", "quit"}:
        break
    state = {"messages": [HumanMessage(content=user_input)]}
    result = travel_info_agent.invoke(state)

    response_msg = result["messages"][-1]
    print(f"Assistant: {response_msg.content}\n")

UK Travel Assistant (type 'exit' to quit)


You:  Suggest beautiful city of England!


Assistant: It's hard to definitively say which city is the "most beautiful" as it's subjective and depends on what you're looking for! However, here are a few cities in England often praised for their beauty, along with some reasons why:

*   **Bath:** Famous for its Roman Baths and stunning Georgian architecture, Bath is a UNESCO World Heritage site. The honey-colored stone buildings give it a unique and elegant feel.
*   **Oxford:** Known as the "City of Dreaming Spires," Oxford is renowned for its historic university buildings, many of which are architectural masterpieces. The city also has beautiful parks and the River Cherwell.
*   **York:** A city with a rich history, York boasts the magnificent York Minster, medieval streets like The Shambles, and well-preserved city walls. It offers a very atmospheric and picturesque experience.
*   **Chester:** This city is unique for its Roman walls, the Rows (two-tiered medieval shopping galleries), and its black-and-white timber-framed buil

You:  Historic architecture. Please search my db


Assistant: I found these places with historic architecture:
* The 15th-17th century Old Town Hall
* The 16th century Old Rectory
* The 17th century Old Grammar School

Would you like to know more about any of these?



You:  exit


# Another Agent could search weather info

## New tool that can get weather info

In [78]:
import random
from typing import Literal, Optional

### A dummy weather info tool
class WeatherForecast(TypedDict):
    town: str
    weather: Literal["sunny", "foggy", "rainy", "windy"]
    temperature: int


class WeatherForecastService:
    _weather_options = ["sunny", "foggy", "rainy", "windy"]
    _temp_min = 18
    _temp_max = 31

    @classmethod
    def get_forecast(cls, town:str) -> Optional[WeatherForecast]:
        weather = random.choice(cls._weather_options)
        temperature = random.randint(cls._temp_min, cls._temp_max)
        return WeatherForecast(town=town, weather=weather, temperature=temperature)

In [79]:
@tool(description="Get the weather forecast, given a town name.")
def weather_forecast(town: str) -> dict:
    """Get a mock weather forecast for a given town. 
    Returns a WeatherForecast object with weather and temperature."""
    forecast = WeatherForecastService.get_forecast(town)
    if forecast is None:
        return {"error": f"No weather data available for '{town}'."}
    return forecast

In [80]:
@tool(description="""Search travel information about destinations in England.""")
def search_travel_info(query: str) -> str: #B
    """Search embedded WikiVoyage content for information about destinations in England."""
    docs = ti_retriever.invoke(query) #C
    top = docs[:4] if isinstance(docs, list) else docs #C
    return "\n---\n".join(d.page_content for d in top) #D

In [81]:
tools = [weather_forecast, search_travel_info]

In [82]:
from langgraph.prebuilt import ToolNode

tools_execution_node = ToolNode(tools)

In [86]:
from langchain_core.messages import SystemMessage

def llm_node(state: AgentState):
    """LLM node that decides whether to call the search tool."""
    current_messages = state["messages"]
    system_message = SystemMessage(
        content="""
        You are a helpful assistant that can search travel information and get the weather forecast.
        Only use the tools to find the information you need (including town names).
        """
    )
    current_messages.append(system_message)
    response_message = llm_with_tools.invoke(current_messages)

    return {"messages": [response_message]}


Same agent graph structure

In [87]:
builder = StateGraph(AgentState)

builder.add_node("llm_node", llm_node)
builder.add_node("tools", tools_execution_node)

builder.add_conditional_edges("llm_node", tools_condition)

builder.add_edge("tools", "llm_node")

builder.set_entry_point("llm_node")

travel_info_agent = builder.compile()

In [88]:
print("UK Travel Assistant (type 'exit' to quit)")
while True:
    user_input = input("You: ").strip()
    if user_input.lower() in {"exit", "quit"}:
        break
    state = {"messages": [HumanMessage(content=user_input)]}
    result = travel_info_agent.invoke(state)

    response_msg = result["messages"][-1]
    print(f"Assistant: {response_msg.content}\n")

UK Travel Assistant (type 'exit' to quit)


You:  Suggest two towns in England for Sunny day.


Assistant: I can suggest Brighton and Bournemouth. Both are coastal towns with beaches, which are great for a sunny day.



You:  Suggest two cities in China for Sunny day.


Assistant: I can only provide information about destinations in England.



You:  exit


In [ ]:
#Can you find a nice seaside Cornwall town with good weather right now and find availability and price for one double hotel room in that town?
